# Tunisair Flight Delay Prediction

Predicting flight delay duration for Tunisair using pre-departure attributes (schedule, route, aircraft), following the same leakage-safe workflow as `03_eda-and-modeling.ipynb`.

## Define the Business Goal

**Stakeholder:** an airline operations team (and, downstream, a travel app surfacing delay risk to passengers).

**Prediction:** the delay duration of a scheduled Tunisair flight, using only information known before departure (schedule, route, aircraft) — never the actual departure/arrival times or recorded delay-reason codes, which only exist after the fact.

**Evaluation metric:** RMSE (minutes), matching the Zindi competition metric. This penalizes large misses more than small ones and stays interpretable in the same units as the target.

This is a **regression** problem: the target `target` is a continuous delay duration.

## Get the Data

Download `train.csv` and `test.csv` from the [Zindi Flight Delay Prediction Challenge](https://zindi.africa/competitions/flight-delay-prediction-challenge/data) and place them in `data/` before running this cell.

In [ ]:
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
train.head()

### First Look at the Data

In [ ]:
train.info()

In [ ]:
# Missingness per column
train.isna().mean().sort_values(ascending=False)

In [ ]:
# Target distribution
train["target"].describe()

## Leakage Audit (Phase 1.6)

Known columns from the Zindi data page: `DATOP`, `FLTID`, `DEPSTN`, `ARRSTN`, `STD`, `STA`, `STATUS`, `ETD`, `ETA`, `ATD`, `ATA`, `DELAY1-4`, `DUR1-4`, `AC`.

| Column | Known before departure? | Notes |
|---|---|---|
| DATOP, FLTID, DEPSTN, ARRSTN, STD, STA, AC | Yes | Schedule/route/aircraft, safe to use |
| ETD, ETA | Check | Estimated times — confirm whether these are set pre-departure or updated in real time |
| ATD, ATA | No | Actual times — only known after the flight, leakage |
| STATUS | Check | Confirm whether this reflects a pre-flight state or a post-hoc outcome |
| DELAY1-4 / DUR1-4 | No | Delay reason codes/durations — recorded after the fact, leakage |

Fill in / correct this table once the real data is loaded and inspected.

## Clean the Data

Leakage-safe steps go here (tidy columns, fix data-entry errors, drop leaky columns) — before the split, following the same discipline as the reference notebook.

## Airport Enrichment (Phase 1.2 / 2.3)

Join `DEPSTN`/`ARRSTN` to airport metadata via `airportsdata`.

In [ ]:
import airportsdata

airports = airportsdata.load("IATA")  # confirm DEPSTN/ARRSTN use IATA codes once data is loaded
# example lookup once codes are confirmed:
# airports["TUN"]

## Train-Test Split

Chronological split (not random) — confirm `DATOP` ordering first (Phase 1.5), then split by a date cutoff so validation data always follows training data in time.

## Baseline Model (Phase 3)

Heuristic baseline (e.g. mean delay, or mean delay per route) scored with RMSE — the number every later model has to beat.

## Model Iteration (Phase 4)

Linear Regression -> Random Forest -> Gradient Boosting, each cross-validated with a time-respecting split (`TimeSeriesSplit`), then hyperparameter tuning on the best performer.

## Error Analysis (Phase 6)

Residuals by route, season, aircraft, and delay magnitude.

## Final Evaluation

Score the selected model once on the held-out test set and save it with `joblib` for use in the Phase 8 web app.